# 04 — Churn & Customer Lifecycle Analysis
**Project:** Olist Brazilian E-Commerce — Customer Behavior & Cohort Analysis

Churn analysis identifies customers who have disengaged from the platform. We define churn as **no purchase activity in 180+ days**, then profile churned vs. active customers to surface patterns that can inform win-back campaigns.

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_raw, build_master
from src.cohort import churn_analysis
from src.utils import apply_style, PALETTE

apply_style()
raw = load_raw()
master = build_master(raw)
print('Data loaded ✓')

## 4.1 Compute Churn (180-Day Threshold)

In [ ]:
churn_df = churn_analysis(master, churn_days=180)

churned_n = churn_df['is_churned'].sum()
active_n  = len(churn_df) - churned_n
churn_rate = churned_n / len(churn_df) * 100

print(f'Total customers    : {len(churn_df):,}')
print(f'Churned (>180 days): {churned_n:,}  ({churn_rate:.1f}%)')
print(f'Active  (≤180 days): {active_n:,}  ({100-churn_rate:.1f}%)')
print()
churn_df.groupby('is_churned')[['total_orders','total_spend','days_since_last_order']].mean().round(2)

## 4.2 Churn Status Breakdown

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pie
axes[0].pie([churned_n, active_n],
            labels=[f'Churned\n({churn_rate:.1f}%)', f'Active\n({100-churn_rate:.1f}%)'],
            colors=['#EF4444', '#10B981'], startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 3},
            autopct='%1.0f%%', pctdistance=0.7, textprops={'fontsize': 12})
axes[0].set_title('Churn Status (180-Day Threshold)', fontsize=12)

# Histogram
axes[1].hist(churn_df['days_since_last_order'], bins=40,
             color=PALETTE['primary'], alpha=0.7, edgecolor='white')
axes[1].axvline(180, color='#EF4444', linestyle='--', linewidth=2, label='Churn threshold (180d)')
axes[1].set_xlabel('Days Since Last Order')
axes[1].set_ylabel('Number of Customers')
axes[1].set_title('Distribution of Days Since Last Purchase', fontsize=12)
axes[1].legend()

fig.suptitle('Customer Churn Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 4.3 Spend Profile — Churned vs Active

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for i, (label, color) in enumerate(zip(['Active', 'Churned'], ['#10B981', '#EF4444'])):
    is_churn = (i == 1)
    subset = churn_df[churn_df['is_churned'] == is_churn]['total_spend']
    subset_clipped = subset[subset <= subset.quantile(0.99)]
    axes[0].hist(subset_clipped, bins=40, alpha=0.6, color=color, label=label, edgecolor='white')

axes[0].set_xlabel('Total Lifetime Spend (R$)')
axes[0].set_ylabel('Number of Customers')
axes[0].set_title('Spend Distribution: Active vs Churned', fontsize=12)
axes[0].legend()

# Revenue at risk
churned_revenue = churn_df[churn_df['is_churned']==1]['total_spend'].sum()
active_revenue  = churn_df[churn_df['is_churned']==0]['total_spend'].sum()
bars = axes[1].bar(['Active', 'Churned'], [active_revenue/1e6, churned_revenue/1e6],
                    color=['#10B981', '#EF4444'], edgecolor='white', width=0.5)
for bar, val in zip(bars, [active_revenue, churned_revenue]):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                 f'R$ {val/1e6:.1f}M', ha='center', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Total Revenue (R$ Millions)')
axes[1].set_title('Historical Revenue: Active vs Churned Customers', fontsize=12)

fig.suptitle('Revenue at Risk from Churned Customers', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Revenue from active customers  : R$ {active_revenue:,.0f}')
print(f'Revenue from churned customers : R$ {churned_revenue:,.0f}')
print(f'Even recovering 10% of churned revenue = R$ {churned_revenue*0.10:,.0f}')

## 4.4 Sensitivity Analysis — Churn Threshold

In [ ]:
thresholds = [60, 90, 120, 150, 180, 240, 365]
results = []
for t in thresholds:
    df = churn_analysis(master, churn_days=t)
    r = df['is_churned'].mean() * 100
    results.append({'threshold': t, 'churn_rate': r})

thresh_df = pd.DataFrame(results)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(thresh_df['threshold'], thresh_df['churn_rate'], 'o-',
        color=PALETTE['danger'] if 'danger' in PALETTE else '#EF4444', lw=2.5, markersize=8)
ax.axvline(180, linestyle='--', color='#6B7280', lw=1.5, label='Selected threshold (180d)')
ax.set_xlabel('Churn Threshold (days)')
ax.set_ylabel('Churn Rate (%)')
ax.set_title('Churn Rate Sensitivity to Threshold Definition', fontsize=12)
ax.legend()
for _, row in thresh_df.iterrows():
    ax.annotate(f"{row['churn_rate']:.0f}%", (row['threshold'], row['churn_rate']),
                textcoords='offset points', xytext=(0, 10), ha='center', fontsize=9)
plt.tight_layout()
plt.show()

### Key Takeaways
- **59.2%** of customers are churned under the 180-day definition — this reflects the platform's low repeat-purchase rate
- Churned customers account for **R$ 8.9M** in historical revenue — a win-back campaign recovering even 5–10% is meaningful
- Average churned customer spent **R$ 162** lifetime vs R$ 169 for active — churn is not driven by low-value customers; high-value customers are churning too
- The sharp cliff in the distribution around days 600–700 reflects the dataset cutoff rather than genuine re-engagement

---
**See also:** The full analysis report at `outputs/reports/olist_customer_analysis_report.html`